In [39]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score, mean_squared_log_error, median_absolute_error, explained_variance_score, max_error, confusion_matrix, f1_score, recall_score, roc_auc_score, roc_curve, precision_score, accuracy_score, classification_report, precision_recall_curve, average_precision_score, matthews_corrcoef
from pathlib import Path

In [60]:
# Iterate over all the files in the predictions folder
inputfolder = Path("../data/Lindsay/week5/predictions/stardist_SR1/")
inputfolder2 = Path("../data/Lindsay/week5/predictions/stardist_SW1/")
inputfolder3 = Path("../data/Lindsay/week5/predictions/stardist_L1/")

for pred in inputfolder.iterdir():
    if str(pred).endswith("_predictions.csv"):
        print(pred)
        try:
            # Read dataframes
            df1 = pd.read_csv(pred)
            df2 = pd.read_csv(inputfolder2 / pred.name)
            df3 = pd.read_csv(inputfolder3 / pred.name)
            
            # Merge dataframes
            merged = pd.merge(df1, df2, on="image")
            merged = pd.merge(merged, df3, on="image")
            
            # Check if the values in micronuclei_x, micronuclei_y, and micronuclei are the same
            merged = merged[merged["micronuclei_x"] == merged["micronuclei_y"]]
            merged = merged[merged["micronuclei_x"] == merged["micronuclei"]]
            merged = merged.drop(columns=["micronuclei_x", "micronuclei_y", 'score', 'score_x', 'score_y'])
            merged = merged.rename(columns={"micronuclei": "count"})
            merged["label"] = (merged["count"]>0).astype(int)
            merged["image"] = merged["image"].apply(lambda x: f"{pred.name[:-16]}_{x}.png")
            
            # Save the merged dataframe
            output = Path(f"../data/Lindsay/week5/merged_predictions/{pred.name}")
            
            # Save the merged dataframe
            merged.to_csv(output, index=False)
            
        except:
            print(f"error with file {pred.name}")


../data/Lindsay/week5/predictions/stardist_SR1/s15c1_predictions.csv
../data/Lindsay/week5/predictions/stardist_SR1/s21c1_predictions.csv
../data/Lindsay/week5/predictions/stardist_SR1/s47c1_predictions.csv
../data/Lindsay/week5/predictions/stardist_SR1/s09c1_predictions.csv
../data/Lindsay/week5/predictions/stardist_SR1/s38c1_predictions.csv
../data/Lindsay/week5/predictions/stardist_SR1/s24c1_predictions.csv
../data/Lindsay/week5/predictions/stardist_SR1/s42c1_predictions.csv
../data/Lindsay/week5/predictions/stardist_SR1/s14c1_predictions.csv
../data/Lindsay/week5/predictions/stardist_SR1/s08c1_predictions.csv
../data/Lindsay/week5/predictions/stardist_SR1/s46c1_predictions.csv
../data/Lindsay/week5/predictions/stardist_SR1/s11c1_predictions.csv
../data/Lindsay/week5/predictions/stardist_SR1/s39c1_predictions.csv
../data/Lindsay/week5/predictions/stardist_SR1/s25c1_predictions.csv
../data/Lindsay/week5/predictions/stardist_SR1/s43c1_predictions.csv
../data/Lindsay/week5/predictions/

In [77]:
# Merge all the result files into one dataframe
inputfolder = Path("../data/Lindsay/week5/merged_predictions/")
predictions = []
for pred in inputfolder.iterdir():
    if str(pred).endswith("_predictions.csv"):
        try:
            # Read dataframes
            df1 = pd.read_csv(pred)
            predictions.append(df1)
        except:
            print(f"error with file {pred.name}")

# Concatenate all the dataframes
all_df = pd.concat(predictions)
all_df.to_csv("../data/Lindsay/week5/merged_predictions_all.csv", index=False)

In [78]:
all_df['count'].value_counts()

count
0    233400
1      5371
2       170
Name: count, dtype: int64

In [79]:
# Balance the dataset so that the number of images with label 0 and 1 are the same
df_0 = all_df[all_df["label"] == 0]
df_1 = all_df[all_df["label"] == 1]
df_0 = df_0.sample(n=len(df_1), random_state=42)
balanced_df = pd.concat([df_0, df_1])
balanced_df.to_csv("../data/Lindsay/week5/merged_predictions_balanced.csv", index=False)

In [80]:
balanced_df['count'].value_counts()

count
0    5541
1    5371
2     170
Name: count, dtype: int64